# CSA0703 — Computer Networks
## Assessment Tool 2 — Industry Problem Solving Task
### Network Data Analysis: Traffic/Latency, Video vs VoIP, and VANET Congestion

**Student Name:** SHANUKA S K  **Reg. No.:** 1911260108

This notebook performs the data analysis behind the submitted findings report (`Findings_Report.docx`). It reads the three provided datasets, computes the statistics referenced in the report, and generates all charts.

Place this notebook in the same folder as the three CSV files before running:
- `Network_Quality_Dataset_dataset.csv`
- `network_traffic_dataset.csv`
- `vanet_traffic_data.csv`


## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option('display.max_columns', None)
plt.rcParams['figure.dpi'] = 110


---
## Q1. Network Quality Dataset — Traffic vs Latency

**Industry scenario:** An organization notices that its network becomes slow when network traffic increases. We investigate whether high traffic is associated with increased latency.

### Load data and combine traffic columns

In [ ]:
df1 = pd.read_csv('Network_Quality_Dataset_dataset.csv', parse_dates=['timestamp'])
df1['total_traffic_bps'] = df1['traffic_in_bps'] + df1['traffic_out_bps']
print("Rows loaded:", len(df1))
df1.head()


### Clean obvious sensor-glitch outliers
A handful of rows report physically implausible traffic values (terabit range). We remove these before analysis.

In [ ]:
before = len(df1)
df1 = df1[df1['total_traffic_bps'] < 1e8].copy()
after = len(df1)
print(f"Removed {before-after} outlier rows; {after} rows retained ({after/before:.2%} of data kept)")


### Summary statistics

In [ ]:
print(df1[['traffic_in_bps','traffic_out_bps','total_traffic_bps','latency_ms']].describe())


### Correlation between traffic and latency

In [ ]:
corr_in = df1['traffic_in_bps'].corr(df1['latency_ms'])
corr_out = df1['traffic_out_bps'].corr(df1['latency_ms'])
corr_total = df1['total_traffic_bps'].corr(df1['latency_ms'])
print("Correlation (traffic_in vs latency):", round(corr_in, 4))
print("Correlation (traffic_out vs latency):", round(corr_out, 4))
print("Correlation (total_traffic vs latency):", round(corr_total, 4))


### Average latency by traffic-load quintile

In [ ]:
df1['traffic_bin'] = pd.qcut(df1['total_traffic_bps'], 5, duplicates='drop')
grp1 = df1.groupby('traffic_bin', observed=True)['latency_ms'].agg(['mean', 'count'])
grp1.index = [f"Q{i+1}" for i in range(len(grp1))]
grp1


### Cross-check: traffic level during the worst latency events

In [ ]:
p90 = df1['latency_ms'].quantile(0.90)
high_lat = df1[df1['latency_ms'] >= p90]
print("90th percentile latency threshold (ms):", p90)
print("Mean total traffic during high-latency events (Mbps):", round(high_lat['total_traffic_bps'].mean()/1e6, 2))
print("Mean total traffic overall (Mbps):", round(df1['total_traffic_bps'].mean()/1e6, 2))

top10 = df1.nlargest(10, 'latency_ms')[['timestamp','total_traffic_bps','latency_ms']]
top10['total_traffic_Mbps'] = (top10['total_traffic_bps']/1e6).round(2)
top10[['timestamp','total_traffic_Mbps','latency_ms']]


### Figure 1 — Traffic and latency over time (line chart)

In [ ]:
df1_sorted = df1.sort_values('timestamp').reset_index(drop=True)
sample = df1_sorted.iloc[::20]  # downsample for readability

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(sample['timestamp'], sample['total_traffic_bps']/1e6, color='#1f77b4', linewidth=0.9)
ax1.set_xlabel('Time')
ax1.set_ylabel('Total Traffic (Mbps)', color='#1f77b4')
ax1.tick_params(axis='y', labelcolor='#1f77b4')

ax2 = ax1.twinx()
ax2.plot(sample['timestamp'], sample['latency_ms'], color='#d62728', alpha=0.6, linewidth=0.9)
ax2.set_ylabel('Latency (ms)', color='#d62728')
ax2.tick_params(axis='y', labelcolor='#d62728')

plt.title('Network Traffic vs Latency Over Time')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


### Figure 2 — Traffic vs latency scatter plot

In [ ]:
plt.figure(figsize=(7, 5.5))
plt.scatter(df1['total_traffic_bps']/1e6, df1['latency_ms'], s=4, alpha=0.12, color='#1f77b4')
plt.xlabel('Total Traffic (Mbps)')
plt.ylabel('Latency (ms)')
plt.title(f'Traffic vs Latency Scatter (Pearson r = {corr_total:.3f})')
plt.tight_layout()
plt.show()


### Figure 3 — Average latency by traffic quintile (bar chart)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(range(len(grp1)), grp1['mean'], color='#2ca02c')
plt.xticks(range(len(grp1)), grp1.index)
plt.xlabel('Traffic Load Quintile (Q1 = lowest ... Q5 = highest)')
plt.ylabel('Average Latency (ms)')
plt.title('Average Latency by Traffic Load Quintile')
for i, v in enumerate(grp1['mean']):
    plt.text(i, v + 1, f"{v:.1f}", ha='center', fontsize=9)
plt.tight_layout()
plt.show()


### Observation
Latency is essentially flat (~94–98 ms) across all traffic-load quintiles, and the correlation coefficient is close to zero. The worst latency events do **not** coincide with high traffic. **Conclusion:** high traffic volume is not the cause of the slowdowns in this dataset — the bottleneck is more likely queuing/jitter or a device-level issue unrelated to bandwidth. **Recommendation:** deploy latency/jitter monitoring correlated with device-level logs rather than assuming a bandwidth upgrade will help; apply QoS traffic prioritization as a low-cost interim measure.

---
## Q2. Network Traffic Dataset — Video Streaming (TCP) vs VoIP (UDP)

**Industry scenario:** A company wants to understand how video-meeting traffic and voice-call traffic differ, to decide which should get priority during congestion.

### Load data and check the Activity_Type / Protocol split

In [ ]:
df2 = pd.read_csv('network_traffic_dataset.csv')
print(df2['Activity_Type'].value_counts())
print()
print(pd.crosstab(df2['Activity_Type'], df2['Protocol']))


In [ ]:
video = df2[df2['Activity_Type'] == 'Video_Streaming']
voip = df2[df2['Activity_Type'] == 'VoIP_Call']

stats2 = df2.groupby('Activity_Type')['Length'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2)
stats2


### Inter-arrival time (regularity of packet timing)

In [ ]:
video_gap = video.sort_values('Time')['Time'].diff().dropna()
voip_gap = voip.sort_values('Time')['Time'].diff().dropna()
print("Video Streaming — mean inter-arrival (s):", round(video_gap.mean(), 4), " std:", round(video_gap.std(), 4))
print("VoIP Call       — mean inter-arrival (s):", round(voip_gap.mean(), 4), " std:", round(voip_gap.std(), 4))


### Figure 4 — Average packet length (bar chart)

In [ ]:
plt.figure(figsize=(6.5, 5))
means = stats2['mean']
bars = plt.bar(means.index, means.values, color=['#1f77b4', '#ff7f0e'])
plt.ylabel('Average Packet Length (bytes)')
plt.title('Average Packet Length: Video Streaming (TCP) vs VoIP (UDP)')
for b, v in zip(bars, means.values):
    plt.text(b.get_x() + b.get_width()/2, v + 10, f"{v:.0f}", ha='center')
plt.tight_layout()
plt.show()


### Figure 5 — Packet length distribution (box plot)

In [ ]:
plt.figure(figsize=(6.5, 5))
plt.boxplot([video['Length'], voip['Length']], tick_labels=['Video Streaming\n(TCP)', 'VoIP Call\n(UDP)'], showfliers=False)
plt.ylabel('Packet Length (bytes)')
plt.title('Packet Length Distribution by Traffic Type')
plt.tight_layout()
plt.show()


### Figure 6 — Packet length distribution (histogram overlay)

In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(video['Length'], bins=40, alpha=0.6, label='Video Streaming (TCP)', color='#1f77b4', density=True)
plt.hist(voip['Length'], bins=40, alpha=0.6, label='VoIP Call (UDP)', color='#ff7f0e', density=True)
plt.xlabel('Packet Length (bytes)')
plt.ylabel('Density')
plt.title('Packet Length Distribution (Normalized)')
plt.legend()
plt.tight_layout()
plt.show()


### Observation
Video Streaming packets average ~1,330 bytes (MTU-sized, bursty arrival) versus VoIP's ~176 bytes (small, regular arrival every ~52 ms). The two traffic types have fundamentally different characteristics: video is throughput-hungry and delay-tolerant (TCP can buffer/retransmit); VoIP is low-bandwidth but delay/jitter-intolerant (UDP, no retransmission). **Recommendation:** give VoIP (UDP) traffic higher priority during congestion (e.g., DiffServ Expedited Forwarding / a low-latency queue), since it has the smaller footprint but the stricter timing requirement.

---
## Q3. VANET Traffic Congestion Dataset

**Industry scenario:** A smart-transportation company wants to know how road congestion affects vehicle-to-vehicle (V2V) communication reliability.

### Load data and check congestion-state labels

In [ ]:
df3 = pd.read_csv('vanet_traffic_data.csv')
order = ['Free-flow', 'Moderate', 'Heavy', 'Gridlock']
df3['label'] = pd.Categorical(df3['label'], categories=order, ordered=True)
print(df3['label'].value_counts().reindex(order))


### Average communication metrics per congestion state

In [ ]:
metrics = ['packet_loss_pct', 'avg_comm_delay_ms', 'queue_length_veh', 'channel_busy_ratio_pct']
grp3 = df3.groupby('label', observed=True)[metrics].mean().reindex(order).round(2)
grp3


### Figure 7 — All four metrics across congestion states (bar charts)

In [ ]:
colors = ['#2ca02c', '#ffbb33', '#ff7f0e', '#d62728']
titles = {
    'packet_loss_pct': 'Average Packet Loss (%)',
    'avg_comm_delay_ms': 'Average Communication Delay (ms)',
    'queue_length_veh': 'Average Queue Length (vehicles)',
    'channel_busy_ratio_pct': 'Average Channel Busy Ratio (%)'
}

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, m in zip(axes.flat, metrics):
    vals = grp3[m]
    bars = ax.bar(order, vals, color=colors)
    ax.set_title(titles[m])
    ax.set_xlabel('Congestion State')
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v, f"{v:.1f}", ha='center', va='bottom', fontsize=9)
plt.suptitle('VANET Communication Performance Across Congestion States', fontsize=13)
plt.tight_layout()
plt.show()


### Figure 8 — Normalized degradation trend (line chart)

In [ ]:
norm = (grp3 - grp3.min()) / (grp3.max() - grp3.min())
plt.figure(figsize=(8, 5.5))
for m in metrics:
    plt.plot(order, norm[m], marker='o', label=titles[m])
plt.xlabel('Congestion State (increasing severity →)')
plt.ylabel('Normalized value (0 = best, 1 = worst)')
plt.title('Relative Degradation Trend Across Congestion States')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


### Observation
Every metric worsens monotonically from Free-flow to Gridlock, with the steepest jump occurring between Heavy and Gridlock (e.g., packet loss roughly doubles again from 15.1% to 29.9%, and delay roughly doubles again from 100.6 ms to 199.3 ms). Channel busy ratio near 90% at Gridlock indicates the wireless medium is close to saturated. **Conclusion:** communication reliability becomes significantly worse at Gridlock, and the wireless channel itself is the bottleneck. **Recommendation:** implement congestion-aware transmit-power and beacon-rate adaptation so vehicles automatically reduce channel usage as channel busy ratio rises, rather than every vehicle transmitting at a fixed rate into an already-saturated medium.

---
## Summary

| Task | Key finding | Recommendation |
|---|---|---|
| Q1 — Traffic vs Latency | No meaningful correlation (r ≈ -0.003) between traffic volume and latency | Investigate queuing/jitter and device-level causes rather than upgrading bandwidth; apply QoS as a low-cost interim step |
| Q2 — Video vs VoIP | Video: large (~1,330 B), bursty, TCP. VoIP: small (~176 B), regular, UDP | Prioritize VoIP (UDP) during congestion — tighter delay/jitter tolerance |
| Q3 — VANET Congestion | All communication metrics degrade sharply, worst at Gridlock (channel busy ratio ~90%) | Congestion-aware beacon-rate / transmit-power adaptation |
